# Hanogt AI eğitimi (Colab)

Bu defter, Hanogt SFT v2 veri setiyle **Qwen3-8B** üzerinde QLoRA eğitimi yapar ve LoRA bağdaştırıcısını Hugging Face hesabındaki özel bir depoya yükler.

1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU.** A100 önerilir; L4 ve T4 çok daha yavaştır.
2. Soldaki 🔑 **Gizli anahtarlar** bölümüne `HF_TOKEN` adıyla **yazma izinli** bir Hugging Face token'ı ekleyin ve deftere erişim verin.
3. Veri seti önce depodan yüklenmiş olmalı (`python3 training/hf_job.py upload --dataset-repo HanStudios/hanogt-sft-v2`).

In [ ]:
!pip install -q "transformers>=5.0" "peft>=0.17" "accelerate>=1.2" "safetensors>=0.4" "bitsandbytes>=0.45" "huggingface_hub>=0.35"

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import snapshot_download

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
DATASET_REPO = "HanStudios/hanogt-sft-v2"          # veri seti ve eğitim kodu (özel)
MODEL_REPO = "HanStudios/hanogt-ai-qwen3-8b-lora"  # bağdaştırıcının gideceği depo (özel)
snapshot_download(DATASET_REPO, repo_type="dataset", local_dir="/content/hanogt", token=os.environ["HF_TOKEN"])

## Veriyi kontrol et

Model yüklemeden örnek ve token sayılarını yazar.

In [ ]:
!cd /content/hanogt && python code/train_lora.py --dry-run --preset small --data train.jsonl --eval-data eval.jsonl

## Eğit

Bir epoch. Bittiğinde bağdaştırıcı `/content/out/adapter` klasörüne yazılır ve `MODEL_REPO` deposuna yüklenir.

In [ ]:
!cd /content/hanogt && python code/train_lora.py --preset small --data train.jsonl --eval-data eval.jsonl --epochs 1 --output /content/out --push-to-hub {MODEL_REPO}

## Dene (düşünerek)

`enable_thinking=True` ile model önce `<think>` içinde düşünür, sonra yanıt verir.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(model, "/content/out/adapter")

messages = [{"role": "user", "content": "Python'da bir listeyi tersine çevirmenin iki yolunu göster."}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=800, do_sample=True, temperature=0.6, top_p=0.95)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))